In [0]:
import uuid
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DateType,
TimestampType, BooleanType, LongType, DoubleType)

CATALOG = "workspace"
DB = f"{CATALOG}.complaints"
RAW_BASE = f"/Volumes/{CATALOG}/complaints/raw_data"

BRONZE = f"{DB}.bronze_complaints"
SILVER = f"{DB}.silver_complaints"
QUARANTINE = f"{DB}.silver_quarantine"
LOGS = f"{DB}.pipeline_execution_logs"
WATERMARK = f"{DB}.pipeline_watermark"

RAW_COLUMNS = ["product", "date_sent_to_company", "issue", "sub_product", "zip_code", "tags",
               "complaint_id", "timely", "company_response", "submitted_via", "company",
               "date_received", "state", "company_public_response", "sub_issue"]
RAW_SCHEMA = StructType([StructField(c, StringType()) for c in RAW_COLUMNS])

BRONZE_SCHEMA = StructType(RAW_SCHEMA.fields + [
    StructField("load_timestamp", TimestampType(), False),
    StructField("source_file", StringType()),
    StructField("batch_id", StringType()),
    StructField("load_type", StringType()),
    StructField("run_date", DateType())])

SILVER_SCHEMA = StructType([
    StructField("complaint_id", StringType(), False),
    StructField("date_received", TimestampType()),
    StructField("date_sent_to_company", TimestampType()),
    StructField("product", StringType()),
    StructField("sub_product", StringType()),
    StructField("issue", StringType()),
    StructField("sub_issue", StringType()),
    StructField("company", StringType()),
    StructField("state", StringType()),
    StructField("zip_code", StringType()),
    StructField("submitted_via", StringType()),
    StructField("company_response", StringType()),
    StructField("company_public_response", StringType()),
    StructField("timely", BooleanType()),
    StructField("tags", StringType()),
    StructField("load_timestamp", TimestampType(), False),
    StructField("batch_id", StringType())])

QUARANTINE_SCHEMA = StructType(RAW_SCHEMA.fields + [
    StructField("quarantine_reason", StringType()),
    StructField("source_file", StringType()),
    StructField("batch_id", StringType()),
    StructField("run_date", DateType()),
    StructField("load_timestamp", TimestampType(), False)])

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("batch_id", StringType()),
    StructField("layer", StringType()),
    StructField("load_type", StringType()),
    StructField("parameter", StringType()),
    StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()),
    StructField("duration_seconds", DoubleType()),
    StructField("status", StringType()),
    StructField("rows_read", LongType()),
    StructField("rows_inserted", LongType()),
    StructField("rows_updated", LongType()),
    StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

WATERMARK_SCHEMA = StructType([
    StructField("layer", StringType(), False),
    StructField("last_processed_date", DateType()),
    StructField("updated_at", TimestampType()),
    StructField("last_batch_id", StringType())])


def now_utc():
    return datetime.now(timezone.utc)


def default_batch_id():
    return f"batch_{now_utc():%Y%m%d_%H%M%S}"


def parse_date(value, default=None):
    value = (value or "").strip()
    return datetime.strptime(value, "%Y-%m-%d").date() if value else default


def create_table(table, schema):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" + ("" if f.nullable else " NOT NULL")
                     for f in schema.fields)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({cols}) USING DELTA")


class PipelineRun:
    """Writes one audit row per run, even on failure."""

    def __init__(self, layer, load_type, parameter, batch_id):
        self.layer, self.load_type, self.parameter, self.batch_id = layer, load_type, parameter, batch_id
        self.run_id = str(uuid.uuid4())
        self.rows_read = self.rows_inserted = self.rows_updated = self.rows_quarantined = 0

    def __enter__(self):
        self.start = now_utc()
        return self

    def __exit__(self, exc_type, exc, tb):
        end = now_utc()
        row = (self.run_id, self.batch_id, self.layer, self.load_type, str(self.parameter),
               self.start, end, (end - self.start).total_seconds(),
               "Failure" if exc_type else "Success",
               int(self.rows_read), int(self.rows_inserted), int(self.rows_updated),
               int(self.rows_quarantined), str(exc)[:4000] if exc_type else None)
        spark.createDataFrame([row], LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOGS)
        return False


def merge_metrics(table):
    """Inserted/updated row counts of the last MERGE on a table."""
    m = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").select("operationMetrics").first()[0]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))


def get_watermark(layer):
    rows = spark.table(WATERMARK).where(F.col("layer") == layer).collect()
    return rows[0]["last_processed_date"] if rows else None


def update_watermark(layer, processed_date, batch_id):
    spark.createDataFrame([(layer, processed_date, now_utc(), batch_id)], WATERMARK_SCHEMA) \
         .createOrReplaceTempView("_wm")
    spark.sql(f"""MERGE INTO {WATERMARK} t USING _wm s ON t.layer = s.layer
                  WHEN MATCHED AND s.last_processed_date >= t.last_processed_date THEN UPDATE SET *
                  WHEN NOT MATCHED THEN INSERT *""")


def dictionary_markdown(schema, primary_key=()):
    print("| Column | Type | Nullable | Key |\n|---|---|---|---|")
    for f in schema.fields:
        print(f"| {f.name} | {f.dataType.simpleString()} | {'yes' if f.nullable else 'no'} | {'PK' if f.name in primary_key else ''} |")

# COMMAND ----------

dbutils.widgets.dropdown("reset_tables", "false", ["false", "true"], "Drop and recreate tables")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.raw_data")
dbutils.fs.mkdirs(f"{RAW_BASE}/full")
dbutils.fs.mkdirs(f"{RAW_BASE}/incremental")

TABLES = {BRONZE: BRONZE_SCHEMA, SILVER: SILVER_SCHEMA, QUARANTINE: QUARANTINE_SCHEMA,
          LOGS: LOG_SCHEMA, WATERMARK: WATERMARK_SCHEMA}

if dbutils.widgets.get("reset_tables") == "true":
    for t in TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {t}")

for t, s in TABLES.items():
    create_table(t, s)

with PipelineRun("Setup", "n/a", "create_tables", default_batch_id()) as run:
    run.rows_read = len(TABLES)

display(spark.table(LOGS))

# COMMAND ----------

dictionary_markdown(BRONZE_SCHEMA)
print()
dictionary_markdown(SILVER_SCHEMA, primary_key=("complaint_id",))